# Phase 1 — QLoRA Fine-tuning: Qwen2.5-VL-3B on ITSEOA Dataset

**Project:** CID — Automated Infrastructure Inspection Commentary Generation  
**Model:** Qwen2.5-VL-3B-Instruct  
**Method:** QLoRA via LLaMA-Factory  
**Dataset:** 214 ITSEOA inspection pairs (image + French commentary)  
**Hardware:** Google Colab T4 GPU (15 GB VRAM)

---

## Purpose

This notebook fine-tunes Qwen2.5-VL-3B-Instruct on the ITSEOA dataset to adapt the model's
language to produce structured French inspection commentaries conforming to ITSEOA/IQOA standards.

## Why LLaMA-Factory

Qwen2.5-VL has a non-standard image preprocessing pipeline and a specific chat template.
LLaMA-Factory handles all model-specific internals via the `qwen2_5_vl` template, letting us
focus on data and hyperparameters rather than fighting architecture details.

## Architecture note — multi-agent context

The adapter produced by this notebook is the core of the **Vision Agent** in the Phase 2
LangGraph pipeline. It will be loaded once at startup and called via a clean
`generate_commentary(image) -> str` interface. The structured output prompt is designed
with this in mind — four labeled sections that the Formatter Agent can parse reliably.

## How to resume after a Colab disconnect

Re-run Sections 0 and 1, then set `RESUME = True` in Section 3 and run from there.

| Section | Description |
|---------|-------------|
| 0 | Environment setup |
| 1 | Configuration — all parameters in one place |
| 2 | Data pipeline — load, convert, register |
| 3 | Training — QLoRA via LLaMA-Factory |
| 4 | Evaluation — ROUGE-L + visual inspection |
| 5 | Export — save adapter and merged model |

---
## Section 0 — Environment Setup

Install dependencies and authenticate. **Restart the runtime after the install cell.**

In [ ]:
import subprocess, sys, os

for pkg in [
    'transformers>=4.49.0', 'accelerate>=0.29.0', 'bitsandbytes>=0.43.0',
    'pymongo', 'python-dotenv', 'rouge-score', 'nltk', 'pandas',
    'pillow', 'scikit-learn',
]:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✓ Dependencies installed')
print('IMPORTANT: Restart the runtime (Runtime > Restart session), then run from Section 0 cell 2.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✓ Drive mounted')

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
print(f'✓ GPU  : {torch.cuda.get_device_name(0)}')
print(f'✓ VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
import transformers; print(f'✓ Transformers : {transformers.__version__}')

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN, add_to_git_credential=False)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('✓ HuggingFace authenticated')
except Exception:
    print('⚠ HF_TOKEN not found — gated models may fail')

In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/CID'
LF_DIR     = f'{DRIVE_BASE}/LLaMA-Factory'
os.makedirs(DRIVE_BASE, exist_ok=True)

if not os.path.exists(LF_DIR):
    print('Cloning LLaMA-Factory v0.9.1...')
    os.system(f'git clone --depth 1 --branch v0.9.1 '
              f'https://github.com/hiyouga/LLaMA-Factory.git {LF_DIR}')
else:
    print('✓ LLaMA-Factory already on Drive')

os.system(f'pip install -q -e "{LF_DIR}[torch,metrics]"')
print('✓ LLaMA-Factory ready')

---
## Section 1 — Configuration

**This is the only cell you need to edit** to control the entire training run.

### Model swappability
To use a different model in the future, change `MODEL_NAME` and `LF_TEMPLATE`.
The rest of the pipeline is model-agnostic — data loading, evaluation, and export
all work the same regardless of which model is selected.

### Structured output design
The `INSTRUCTION` uses a fixed 4-section format. This is intentional for Phase 2:
the Formatter Agent in the LangGraph pipeline will parse these sections by label.
Free-form output would require fragile regex parsing. Structured output makes
the downstream pipeline robust.

### QLoRA parameters
- `LORA_R=16`: standard rank for domain adaptation. Balances capacity vs overfitting risk.
- `LORA_ALPHA=32`: scaling = 2×rank, standard convention.
- `freeze_vision_tower=true`: critical with small datasets — the vision encoder already
  extracts structural features; only the language side needs domain adaptation.

In [ ]:
import os
from google.colab import userdata

# ── MongoDB ────────────────────────────────────────────────────────────────
try:    MONGO_URI = userdata.get('MONGO_URI')
except: MONGO_URI = ''
MONGO_DB, MONGO_COLL = 'inspection_db', 'itseoa_pairs'

# ── Paths ──────────────────────────────────────────────────────────────────
DRIVE_BASE    = '/content/drive/MyDrive/CID'
LF_DIR        = f'{DRIVE_BASE}/LLaMA-Factory'
PHASE1_DIR    = f'{DRIVE_BASE}/phase1'
DATA_DIR      = f'{LF_DIR}/data'
OUTPUT_DIR    = f'{PHASE1_DIR}/output'
JSON_FALLBACK = f'{DRIVE_BASE}/data/finetuning_dataset.json'
DATASET_NAME  = 'cid_itseoa'

# ── Model — change these two lines to swap to a different model ────────────
MODEL_NAME  = 'Qwen/Qwen2.5-VL-3B-Instruct'
LF_TEMPLATE = 'qwen2_5_vl'

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LORA_TARGET = 'all'

# ── Training ───────────────────────────────────────────────────────────────
NUM_EPOCHS    = 3
BATCH_SIZE    = 2
GRAD_ACCUM    = 8
LEARNING_RATE = '2e-4'
CUTOFF_LEN    = 2048
WARMUP_RATIO  = 0.03
SAVE_STEPS    = 50
VAL_SPLIT     = 0.1
RANDOM_SEED   = 42

# ── Structured output prompt ───────────────────────────────────────────────
# Fixed 4-section format for Phase 2 agent parseability.
INSTRUCTION = (
    "You are a civil engineering expert specializing in infrastructure inspection. "
    "Analyze this image and write a professional inspection commentary in French "
    "following the ITSEOA standard structure:\n"
    "- Désordre observé: [visual description of the defect]\n"
    "- Causes probables: [likely causes]\n"
    "- Gravité: [severity assessment]\n"
    "- Suites à donner: [recommended actions]\n"
    "Only describe what is visually observable in the image."
)

for d in [PHASE1_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print('✓ Configuration loaded')
print(f'  Model          : {MODEL_NAME}')
print(f'  Template       : {LF_TEMPLATE}')
print(f'  MongoDB        : {"set" if MONGO_URI else "not set — JSON fallback"}')
print(f'  LoRA rank      : {LORA_R}')
print(f'  Effective batch: {BATCH_SIZE * GRAD_ACCUM}')

---
## Section 2 — Data Pipeline

### What happens here
1. Load 214 ITSEOA pairs from MongoDB or JSON fallback
2. Save images to disk (LLaMA-Factory reads from file paths, not base64)
3. Convert to ShareGPT format with structured prompt
4. Split 90/10 into train/validation
5. Register with LLaMA-Factory's dataset registry

### ShareGPT format (LLaMA-Factory v0.9.1)
```json
{
  "conversations": [
    {"from": "human", "value": "<image>\nAnalyze this image..."},
    {"from": "gpt",   "value": "Désordre observé: ..."}
  ],
  "images": ["/path/to/image.jpg"]
}
```
Note: v0.9.1 uses `from`/`value` keys, not `role`/`content`.

In [ ]:
import json, base64, os
from io import BytesIO
from PIL import Image
from sklearn.model_selection import train_test_split

def load_records():
    if MONGO_URI:
        try:
            from pymongo import MongoClient
            client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
            client.admin.command('ping')
            records = list(client[MONGO_DB][MONGO_COLL].find({}))
            print(f'✓ MongoDB: {len(records)} records'); return records
        except Exception as e:
            print(f'⚠ MongoDB failed ({e}) — using JSON')
    with open(JSON_FALLBACK, encoding='utf-8') as f:
        records = json.load(f)
    print(f'✓ JSON: {len(records)} records'); return records

records = load_records()

In [ ]:
IMAGES_DIR = f'{DATA_DIR}/cid_itseoa_images'
os.makedirs(IMAGES_DIR, exist_ok=True)

data, skipped = [], 0

for i, r in enumerate(records):
    commentary = r.get('commentary_full') or r.get('response', '')
    image_b64  = r.get('image_b64', '')
    if not image_b64 or not commentary:
        skipped += 1; continue

    img_path = f'{IMAGES_DIR}/sample_{i:04d}.jpg'
    if not os.path.exists(img_path):
        img = Image.open(BytesIO(base64.b64decode(image_b64))).convert('RGB')
        img.save(img_path, 'JPEG', quality=95)

    data.append({
        'conversations': [
            {'from': 'human', 'value': f'<image>\n{INSTRUCTION}'},
            {'from': 'gpt',   'value': commentary},
        ],
        'images': [img_path],
    })

print(f'✓ {len(data)} samples converted ({skipped} skipped)')

train_data, val_data = train_test_split(
    data, test_size=VAL_SPLIT, random_state=RANDOM_SEED, shuffle=True
)
print(f'  Train: {len(train_data)}  |  Val: {len(val_data)}')

TRAIN_FILE = f'{DATA_DIR}/{DATASET_NAME}_train.json'
VAL_FILE   = f'{DATA_DIR}/{DATASET_NAME}_val.json'

for path, split in [(TRAIN_FILE, train_data), (VAL_FILE, val_data)]:
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(split, f, ensure_ascii=False, indent=2)

print(f'✓ Saved: {TRAIN_FILE}')
print(f'✓ Saved: {VAL_FILE}')

In [ ]:
dataset_info_path = f'{DATA_DIR}/dataset_info.json'
if os.path.exists(dataset_info_path):
    with open(dataset_info_path) as f: dataset_info = json.load(f)
else:
    dataset_info = {}

for split, fname in [('train', f'{DATASET_NAME}_train.json'),
                     ('val',   f'{DATASET_NAME}_val.json')]:
    dataset_info[f'{DATASET_NAME}_{split}'] = {
        'file_name' : fname,
        'formatting': 'sharegpt',
        'columns'   : {'messages': 'conversations', 'images': 'images'},
    }

with open(dataset_info_path, 'w') as f:
    json.dump(dataset_info, f, indent=2)

missing = [s for s in train_data+val_data if not os.path.exists(s['images'][0])]
if missing:
    print(f'✗ {len(missing)} images missing — re-run the conversion cell')
else:
    print(f'✓ Registered: {DATASET_NAME}_train / {DATASET_NAME}_val')
    print(f'✓ All {len(train_data)+len(val_data)} images verified on disk')
    print('  Ready to train.')

---
## Section 3 — Training

### What LLaMA-Factory does internally
1. Loads Qwen2.5-VL-3B in 4-bit NF4 quantization (~2.4 GB VRAM)
2. Applies QLoRA adapters to all linear layers
3. For each batch:
   - Images → tiled at 448×448 → vision encoder → 256 visual tokens
   - Visual tokens + text tokens → language model
   - Loss computed **only on assistant response tokens** (commentary sections)
   - Gradients flow through LoRA matrices only (vision encoder stays frozen)
4. Saves checkpoint every 50 steps to Drive
5. Keeps the checkpoint with the best validation loss

### Resume after disconnect
Set `RESUME = True` below and re-run this section.
LLaMA-Factory will pick up from the last checkpoint automatically.

### Expected runtime on T4
~60–90 minutes for 3 epochs on ~192 training samples.

In [ ]:
# Set RESUME = True if resuming after a Colab disconnect
RESUME = False

yaml_content = f"""### Model
model_name_or_path: {MODEL_NAME}
trust_remote_code: true

### Training method
stage: sft
do_train: true
finetuning_type: lora

### QLoRA
quantization_bit: 4
quantization_type: nf4

### LoRA
lora_rank: {LORA_R}
lora_alpha: {LORA_ALPHA}
lora_dropout: {LORA_DROPOUT}
lora_target: {LORA_TARGET}

### Vision encoder — frozen (214 samples too few to train it)
freeze_vision_tower: true

### Dataset
dataset: {DATASET_NAME}_train
eval_dataset: {DATASET_NAME}_val
template: {LF_TEMPLATE}
cutoff_len: {CUTOFF_LEN}
val_size: 0

### Output
output_dir: {OUTPUT_DIR}
logging_dir: {OUTPUT_DIR}/logs
save_steps: {SAVE_STEPS}
save_total_limit: 3
logging_steps: 10

### Hyperparameters
per_device_train_batch_size: {BATCH_SIZE}
per_device_eval_batch_size: 1
gradient_accumulation_steps: {GRAD_ACCUM}
num_train_epochs: {NUM_EPOCHS}
learning_rate: {LEARNING_RATE}
lr_scheduler_type: cosine
warmup_ratio: {WARMUP_RATIO}
max_grad_norm: 0.3
weight_decay: 0.001

### Precision
bf16: true

### Evaluation
eval_strategy: steps
eval_steps: {SAVE_STEPS}
save_strategy: steps
load_best_model_at_end: true
metric_for_best_model: eval_loss

### Resume
resume_lora_training: {str(RESUME).lower()}

### Misc
seed: {RANDOM_SEED}
report_to: none
overwrite_output_dir: true
dataloader_pin_memory: false
"""

YAML_PATH = f'{PHASE1_DIR}/train_config.yaml'
with open(YAML_PATH, 'w') as f:
    f.write(yaml_content)

steps_per_epoch = len(train_data) // (BATCH_SIZE * GRAD_ACCUM)
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f'✓ Config saved: {YAML_PATH}')
print(f'  Resume       : {RESUME}')
print(f'  ~{total_steps} total steps  |  ~{total_steps*25//60}–{total_steps*35//60} min on T4')

In [ ]:
import os
os.chdir(LF_DIR)
get_ipython().system(f'llamafactory-cli train {YAML_PATH}')

---
## Section 4 — Evaluation

### Two-track evaluation

**ROUGE-L (quantitative):** measures lexical similarity between generated and expert
commentary. Zero-shot baseline from Phase 0: 0.1452. Expected improvement after
fine-tuning on 214 pairs: 0.25–0.40.

**Visual inspection (qualitative):** the only way to know if the model is actually
useful. We look for: correct ITSEOA structure (4 labeled sections), domain vocabulary
(fissuration, éclatement, armatures, etc.), and relevance to the image content.

In [ ]:
import json, os
from PIL import Image
from IPython.display import display
import textwrap, subprocess
from rouge_score import rouge_scorer as rs_module

with open(VAL_FILE, encoding='utf-8') as f:
    val_samples = json.load(f)
print(f'✓ {len(val_samples)} validation samples')

# Write inference script to avoid shell escaping issues
inference_script = f"""
import sys
sys.path.insert(0, '{LF_DIR}')
from llamafactory.chat import ChatModel
import json

model = ChatModel(dict(
    model_name_or_path   = {json.dumps(MODEL_NAME)},
    adapter_name_or_path = {json.dumps(OUTPUT_DIR)},
    template             = {json.dumps(LF_TEMPLATE)},
    finetuning_type      = 'lora',
    trust_remote_code    = True,
    quantization_bit     = 4,
))

img_path    = sys.argv[1]
instruction = {json.dumps(INSTRUCTION)}
response    = model.chat([{{'role':'user','content':instruction,'images':[img_path]}}])
print(response[0].response_text)
"""

script_path = f'{PHASE1_DIR}/run_inference.py'
with open(script_path, 'w', encoding='utf-8') as f:
    f.write(inference_script)
print(f'✓ Inference script ready')

In [ ]:
r_scorer = rs_module.RougeScorer(['rougeL'], use_stemmer=False)

def rouge_l(ref, hyp):
    if not hyp.strip(): return 0.0
    return round(r_scorer.score(ref, hyp)['rougeL'].fmeasure, 4)

print('Running ROUGE-L evaluation on validation set...')
print()

eval_results = []
for i, sample in enumerate(val_samples):
    img_path  = sample['images'][0]
    reference = sample['conversations'][1]['value']

    result = subprocess.run(
        ['python', script_path, img_path],
        capture_output=True, text=True, timeout=120
    )
    generated = result.stdout.strip() if result.returncode == 0 else ''
    rl = rouge_l(reference, generated)
    eval_results.append({'rl': rl, 'generated': generated,
                         'reference': reference, 'img': img_path})
    print(f'  [{i+1:2d}/{len(val_samples)}] ROUGE-L: {rl:.4f}')

avg_rl = sum(r['rl'] for r in eval_results) / len(eval_results)
baseline = 0.1452
print()
print(f'Average ROUGE-L : {avg_rl:.4f}')
print(f'Zero-shot (P0)  : {baseline:.4f}')
print(f'Improvement     : {((avg_rl-baseline)/baseline*100):+.1f}%')

In [ ]:
NUM_DISPLAY = 5

print('=' * 70)
print('QUALITATIVE EVALUATION — Generated vs Expert Commentary')
print('=' * 70)

for i, r in enumerate(eval_results[:NUM_DISPLAY]):
    print(f'\n── Sample {i+1}/{NUM_DISPLAY}  |  ROUGE-L: {r["rl"]:.4f} ──────────────────────')
    img = Image.open(r['img']); img.thumbnail((380,380)); display(img)

    print('\nGENERATED:')
    print(textwrap.fill(r['generated'][:600], width=68,
                        initial_indent='  ', subsequent_indent='  '))
    if len(r['generated'])>600: print('  [...]')

    print('\nEXPERT REFERENCE:')
    print(textwrap.fill(r['reference'][:500], width=68,
                        initial_indent='  ', subsequent_indent='  '))
    if len(r['reference'])>500: print('  [...]')

saved = {
    'avg_rouge_l': avg_rl, 'baseline': baseline,
    'improvement_pct': (avg_rl-baseline)/baseline*100,
    'n_samples': len(eval_results),
    'per_sample': [{'rl': r['rl'], 'img': r['img']} for r in eval_results],
}
with open(f'{PHASE1_DIR}/eval_results.json','w') as f:
    json.dump(saved, f, indent=2)
print(f'\n✓ Eval results saved: {PHASE1_DIR}/eval_results.json')

---
## Section 5 — Export

The adapter is already saved in `OUTPUT_DIR` by the trainer.
Here we also produce a merged model (base + adapter fused) for Phase 2 use.

**Why merge?** The LangGraph Vision Agent loads the model once at startup.
A merged model has no PEFT dependency and is faster to load and run.
The merge formula: `W_final = W_base + (alpha/rank) × A × B`

In [ ]:
import os
os.chdir(LF_DIR)

MERGED_DIR = f'{PHASE1_DIR}/merged_model'
os.makedirs(MERGED_DIR, exist_ok=True)

export_yaml = f"""model_name_or_path: {MODEL_NAME}
adapter_name_or_path: {OUTPUT_DIR}
template: {LF_TEMPLATE}
finetuning_type: lora
trust_remote_code: true
export_dir: {MERGED_DIR}
export_size: 4
export_device: cpu
export_legacy_format: false
"""

EXPORT_YAML = f'{PHASE1_DIR}/export_config.yaml'
with open(EXPORT_YAML, 'w') as f:
    f.write(export_yaml)

print('Merging adapter into base model (~5 min)...')
get_ipython().system(f'llamafactory-cli export {EXPORT_YAML}')
print(f'\n✓ Merged model: {MERGED_DIR}')

In [ ]:
print('=' * 60)
print('PHASE 1 COMPLETE')
print('=' * 60)
print(f'Model          : {MODEL_NAME}')
print(f'Method         : QLoRA (rank={LORA_R}, alpha={LORA_ALPHA})')
print(f'Training data  : {len(train_data)} samples')
print(f'Validation     : {len(val_data)} samples')
print(f'Epochs         : {NUM_EPOCHS}')
print()
print(f'ROUGE-L baseline (zero-shot) : 0.1452')
print(f'ROUGE-L after fine-tuning    : {avg_rl:.4f}')
print(f'Improvement                  : {((avg_rl-0.1452)/0.1452*100):+.1f}%')
print()
print('Saved to Google Drive:')
print(f'  Adapter      : {OUTPUT_DIR}')
print(f'  Merged model : {MERGED_DIR}')
print(f'  Eval results : {PHASE1_DIR}/eval_results.json')
print(f'  Train config : {YAML_PATH}')
print()
print('Next: Phase 2 — LangGraph agentic pipeline')
print('  The merged model serves as the Vision Agent core.')